# Graph-aware retrieval
Oracle AI Agent Memory

In [ ]:
import importlib
import pandas as pd
from IPython.display import display
import demo_helpers

importlib.reload(demo_helpers)
from demo_helpers import (
    RUN_ID, USER_ID, AGENT_ID,
    start_demo, reset_demo, reset_links, direct_table, show_comparison, cleanup,
)

memory, db_pool = await start_demo(globals().get('db_pool'))

## 1. Store three memories

In [ ]:
await reset_demo(memory)

facts = {
    'A': 'The project review is on Monday.',
    'B': 'The project review has moved to Tuesday.',
    'C': 'The Tuesday project review is at 10 a.m. in Room 3.',
}
memory_ids = {}
for label, text in facts.items():
    memory_ids[label] = await memory.add_memory_async(
        text,
        memory_id=f'{RUN_ID}-{label}',
        memory_type='fact',
        user_id=USER_ID,
        agent_id=AGENT_ID,
        metadata={'demo': 'project-review', 'label': label},
        autonomous_linking=False,
        ttl_days=None,
    )

label_by_id = {record_id: label for label, record_id in memory_ids.items()}
display(pd.DataFrame([{'Record': label, 'Memory': text} for label, text in facts.items()]))

## 2. Search by similarity

In [ ]:
QUERY = 'When and where is the project review, and what changed?'

scope = {
    'user_id': USER_ID,
    'agent_id': AGENT_ID,
    'exact_user_match': True,
    'exact_agent_match': True,
    'record_types': ['fact'],
}
unlinked_results = await memory.search_async(
    QUERY, **scope, max_results=3, num_hops=0, include_invalid_results=False,
)
display(direct_table(unlinked_results, label_by_id))

## 3. Connect the facts
**C —refines→ B —supersedes→ A**

The older records become historical; they are not deleted.

In [ ]:
await reset_links(memory)

await memory.link_records_async(
    source_record_id=memory_ids['B'],
    source_record_type='fact',
    target_record_id=memory_ids['A'],
    target_record_type='fact',
    relation_type='supersedes',
    relation_id=f'{RUN_ID}-supersedes',
)
await memory.link_records_async(
    source_record_id=memory_ids['C'],
    source_record_type='fact',
    target_record_id=memory_ids['B'],
    target_record_type='fact',
    relation_type='refines',
    relation_id=f'{RUN_ID}-refines',
)

## 4. Compare retrieval
Same query and filters. Only `num_hops` changes.

In [ ]:
search_options = dict(
    **scope,
    max_results=1,
    max_linked_results=2,
    include_invalid_results=False,
)

flat_results = await memory.search_async(QUERY, **search_options, num_hops=0)
graph_results = await memory.search_async(QUERY, **search_options, num_hops=2)

show_comparison(flat_results, graph_results, label_by_id)

In [ ]:
DELETE_THIS_RUN = True
await cleanup(memory, db_pool, delete=DELETE_THIS_RUN)